### qwen-model

In [ ]:
## write the full model architecture here

In [86]:
## rope
import torch

def compute_rope_angles(head_dim, theta_base=10000, context_length=2048, dtype=torch.float32):
    
    assert head_dim // 2 == 0 ## head_dim should be  even

    index = torch.arange(0, head_dim, 2, dtype=dtype)
    inv_freq = 1.0 / (theta_base ** (2 * index / head_dim))

    ## compute positions
    positions = torch.arange(context_length, dtype=dtype) ## we are calculating the positions here  

    ## computing the angle 
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0) ### positions = 2048, 1  ** 1 ---- inv_freq = head_dim // 2 -----> angles = 2048 * 64

   
    cos = torch.cos(angles) ## 2048, 64
    sin = torch.sin(angles)  ## 2048, 64




In [ ]:
def apply_rope(x, cos, sin):

    B, T, H, D = x.shape

    x1 = x[..., ::2]   # [B, T, H, D/2]
    x2 = x[..., 1::2]  # [B, T, H, D/2]

    # Handle both training and inference
    if cos.dim() == 2:  # training
        cos = cos.unsqueeze(0).unsqueeze(2)  # [1, T, 1, D/2]
        sin = sin.unsqueeze(0).unsqueeze(2)
    else:               # inference (single position)
        cos = cos[None, None, None, :]       # [1, 1, 1, D/2]
        sin = sin[None, None, None, :]

    x_even = x1 * cos - x2 * sin
    x_odd  = x1 * sin + x2 * cos

    x_out = torch.stack([x_even, x_odd], dim=-1).flatten(-2)

    return x_out

In [88]:
context_length = 2048

positions = torch.arange(context_length, dtype=torch.float32)
positions  = positions.unsqueeze(1)
print(positions)
print(positions.shape)

tensor([[0.0000e+00],
        [1.0000e+00],
        [2.0000e+00],
        ...,
        [2.0450e+03],
        [2.0460e+03],
        [2.0470e+03]])
torch.Size([2048, 1])


In [89]:
import torch
head_dim = 128
pairs =  torch.arange(0, head_dim//2, dtype=torch.float32) 

theta_base = 10000
inv_freq = 1.0 / (theta_base ** (2* pairs / head_dim))



print(f"these are inv_frequencies: {inv_freq}")

inv_freq = inv_freq.unsqueeze(0)
print(f"{inv_freq.shape}")





these are inv_frequencies: tensor([1.0000e+00, 8.6596e-01, 7.4989e-01, 6.4938e-01, 5.6234e-01, 4.8697e-01,
        4.2170e-01, 3.6517e-01, 3.1623e-01, 2.7384e-01, 2.3714e-01, 2.0535e-01,
        1.7783e-01, 1.5399e-01, 1.3335e-01, 1.1548e-01, 1.0000e-01, 8.6596e-02,
        7.4989e-02, 6.4938e-02, 5.6234e-02, 4.8697e-02, 4.2170e-02, 3.6517e-02,
        3.1623e-02, 2.7384e-02, 2.3714e-02, 2.0535e-02, 1.7783e-02, 1.5399e-02,
        1.3335e-02, 1.1548e-02, 1.0000e-02, 8.6596e-03, 7.4989e-03, 6.4938e-03,
        5.6234e-03, 4.8697e-03, 4.2170e-03, 3.6517e-03, 3.1623e-03, 2.7384e-03,
        2.3714e-03, 2.0535e-03, 1.7783e-03, 1.5399e-03, 1.3335e-03, 1.1548e-03,
        1.0000e-03, 8.6596e-04, 7.4989e-04, 6.4938e-04, 5.6234e-04, 4.8697e-04,
        4.2170e-04, 3.6517e-04, 3.1623e-04, 2.7384e-04, 2.3714e-04, 2.0535e-04,
        1.7783e-04, 1.5399e-04, 1.3335e-04, 1.1548e-04])
torch.Size([1, 64])


In [90]:
angles = positions *  inv_freq # positions.shape = 2048,1  and inv_freq.shape = 64, 1
print(angles)
print(angles.shape)  ## 2048, 64

tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [1.0000e+00, 8.6596e-01, 7.4989e-01,  ..., 1.5399e-04, 1.3335e-04,
         1.1548e-04],
        [2.0000e+00, 1.7319e+00, 1.4998e+00,  ..., 3.0799e-04, 2.6670e-04,
         2.3096e-04],
        ...,
        [2.0450e+03, 1.7709e+03, 1.5335e+03,  ..., 3.1491e-01, 2.7271e-01,
         2.3615e-01],
        [2.0460e+03, 1.7718e+03, 1.5343e+03,  ..., 3.1507e-01, 2.7284e-01,
         2.3627e-01],
        [2.0470e+03, 1.7726e+03, 1.5350e+03,  ..., 3.1522e-01, 2.7297e-01,
         2.3638e-01]])
torch.Size([2048, 64])


In [91]:
cos = torch.cos(angles)
print(cos)
print(cos.shape)

print("---------")
sin = torch.sin(angles)
print(sin)
print(sin.shape)

tensor([[ 1.0000,  1.0000,  1.0000,  ...,  1.0000,  1.0000,  1.0000],
        [ 0.5403,  0.6479,  0.7318,  ...,  1.0000,  1.0000,  1.0000],
        [-0.4161, -0.1604,  0.0709,  ...,  1.0000,  1.0000,  1.0000],
        ...,
        [-0.9844,  0.5726,  0.9062,  ...,  0.9508,  0.9630,  0.9722],
        [-0.6799,  0.9955,  0.3750,  ...,  0.9508,  0.9630,  0.9722],
        [ 0.2497,  0.7174, -0.3574,  ...,  0.9507,  0.9630,  0.9722]])
torch.Size([2048, 64])
---------
tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 8.4147e-01,  7.6172e-01,  6.8156e-01,  ...,  1.5399e-04,
          1.3335e-04,  1.1548e-04],
        [ 9.0930e-01,  9.8705e-01,  9.9748e-01,  ...,  3.0799e-04,
          2.6670e-04,  2.3096e-04],
        ...,
        [ 1.7590e-01, -8.1986e-01,  4.2275e-01,  ...,  3.0974e-01,
          2.6934e-01,  2.3396e-01],
        [-7.3331e-01, -9.5051e-02,  9.2701e-01,  ...,  3.0988e-01,
          2.6947e-01,  2.3408e-01],
     

In [92]:

pairs

tensor([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
        14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25., 26., 27.,
        28., 29., 30., 31., 32., 33., 34., 35., 36., 37., 38., 39., 40., 41.,
        42., 43., 44., 45., 46., 47., 48., 49., 50., 51., 52., 53., 54., 55.,
        56., 57., 58., 59., 60., 61., 62., 63.])

In [93]:
cos
print(cos.shape)

torch.Size([2048, 64])


In [94]:
sin
print(sin.shape)

torch.Size([2048, 64])


In [95]:
head_dim = 128
q = torch.arange(0,head_dim, dtype=torch.float32)
q = q.unsqueeze(0).unsqueeze(0).unsqueeze(0)

print(q)
print(q.shape)


tensor([[[[  0.,   1.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.,  10.,
            11.,  12.,  13.,  14.,  15.,  16.,  17.,  18.,  19.,  20.,  21.,
            22.,  23.,  24.,  25.,  26.,  27.,  28.,  29.,  30.,  31.,  32.,
            33.,  34.,  35.,  36.,  37.,  38.,  39.,  40.,  41.,  42.,  43.,
            44.,  45.,  46.,  47.,  48.,  49.,  50.,  51.,  52.,  53.,  54.,
            55.,  56.,  57.,  58.,  59.,  60.,  61.,  62.,  63.,  64.,  65.,
            66.,  67.,  68.,  69.,  70.,  71.,  72.,  73.,  74.,  75.,  76.,
            77.,  78.,  79.,  80.,  81.,  82.,  83.,  84.,  85.,  86.,  87.,
            88.,  89.,  90.,  91.,  92.,  93.,  94.,  95.,  96.,  97.,  98.,
            99., 100., 101., 102., 103., 104., 105., 106., 107., 108., 109.,
           110., 111., 112., 113., 114., 115., 116., 117., 118., 119., 120.,
           121., 122., 123., 124., 125., 126., 127.]]]])
torch.Size([1, 1, 1, 128])


In [96]:
k = torch.arange(0,head_dim, dtype=torch.float32)
k = k.unsqueeze(0).unsqueeze(0).unsqueeze(0)

print(k)
print(k.shape)


tensor([[[[  0.,   1.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.,  10.,
            11.,  12.,  13.,  14.,  15.,  16.,  17.,  18.,  19.,  20.,  21.,
            22.,  23.,  24.,  25.,  26.,  27.,  28.,  29.,  30.,  31.,  32.,
            33.,  34.,  35.,  36.,  37.,  38.,  39.,  40.,  41.,  42.,  43.,
            44.,  45.,  46.,  47.,  48.,  49.,  50.,  51.,  52.,  53.,  54.,
            55.,  56.,  57.,  58.,  59.,  60.,  61.,  62.,  63.,  64.,  65.,
            66.,  67.,  68.,  69.,  70.,  71.,  72.,  73.,  74.,  75.,  76.,
            77.,  78.,  79.,  80.,  81.,  82.,  83.,  84.,  85.,  86.,  87.,
            88.,  89.,  90.,  91.,  92.,  93.,  94.,  95.,  96.,  97.,  98.,
            99., 100., 101., 102., 103., 104., 105., 106., 107., 108., 109.,
           110., 111., 112., 113., 114., 115., 116., 117., 118., 119., 120.,
           121., 122., 123., 124., 125., 126., 127.]]]])
torch.Size([1, 1, 1, 128])


In [97]:
x1 = q[..., ::2]
print(x1)
print(x1.shape)

tensor([[[[  0.,   2.,   4.,   6.,   8.,  10.,  12.,  14.,  16.,  18.,  20.,
            22.,  24.,  26.,  28.,  30.,  32.,  34.,  36.,  38.,  40.,  42.,
            44.,  46.,  48.,  50.,  52.,  54.,  56.,  58.,  60.,  62.,  64.,
            66.,  68.,  70.,  72.,  74.,  76.,  78.,  80.,  82.,  84.,  86.,
            88.,  90.,  92.,  94.,  96.,  98., 100., 102., 104., 106., 108.,
           110., 112., 114., 116., 118., 120., 122., 124., 126.]]]])
torch.Size([1, 1, 1, 64])


In [98]:
x2 = k[..., ::2]
print(x2)
print(x2.shape)

tensor([[[[  0.,   2.,   4.,   6.,   8.,  10.,  12.,  14.,  16.,  18.,  20.,
            22.,  24.,  26.,  28.,  30.,  32.,  34.,  36.,  38.,  40.,  42.,
            44.,  46.,  48.,  50.,  52.,  54.,  56.,  58.,  60.,  62.,  64.,
            66.,  68.,  70.,  72.,  74.,  76.,  78.,  80.,  82.,  84.,  86.,
            88.,  90.,  92.,  94.,  96.,  98., 100., 102., 104., 106., 108.,
           110., 112., 114., 116., 118., 120., 122., 124., 126.]]]])
torch.Size([1, 1, 1, 64])


In [99]:
print(q.shape)
cos = cos[None,:, None,:]
print(cos.shape)

sin = sin[None,:,None,:]
print(sin.shape)


torch.Size([1, 1, 1, 128])
torch.Size([1, 2048, 1, 64])
torch.Size([1, 2048, 1, 64])


In [109]:
x = torch.arange(0,10,2)
y = torch.arange(1,10,2)

In [110]:
x

tensor([0, 2, 4, 6, 8])

In [111]:
y

tensor([1, 3, 5, 7, 9])

In [113]:
stack = torch.stack([x,y], dim=-1)
stack

tensor([[0, 1],
        [2, 3],
        [4, 5],
        [6, 7],
        [8, 9]])

In [ ]:
torch.flatten(stack, )